Predictions.csv lacks the high-dimensional latent vectors needed for our UMAP projection, so I extract these using the pretrained AkidaNet.

In [1]:
import os
os.environ["TF_USE_LEGACY_KERAS"] = "1"
import numpy as np
import pandas as pd
import tensorflow as tf
import akida
from akida_models import akidanet_imagenet_pretrained
import cnn2snn
import quantizeml.models
from tqdm import tqdm


2026-11-09 17:12:53.058786: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-11-09 17:12:53.207663: I external/local_xla/xla/tsl/cuda/cudart_stub.cc:32] Could not find cuda drivers on your machine, GPU will not be used.
2026-11-09 17:12:53.276306: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1794240773.381121   67945 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1794240773.407590   67945 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1794240773.611945   67945 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linkin

In [2]:


cnn2snn.set_akida_version(cnn2snn.AkidaVersion.v1) #Akida 1.0 compatibility as requested by Yoshi

model = akidanet_imagenet_pretrained(0.5, quantized=False, bitwidth=4) #pretrained, quantized model


# isolate the penultimate layer
# model.layers[-2] = grab features before 1000-class softmax
feat_extractor = tf.keras.Model(inputs=model.input, outputs=model.layers[-2].input)


df = pd.read_csv("results_card/predictions.csv")

df["keras_correct"] = df["label"] == df["keras_pred"]
df["akida_correct"] = df["label"] == df["akida_pred"]

#subset because CPU takes 7h
df_subset = df.groupby(["corruption", "severity"]).sample(n=100, random_state=42).copy()


def extract_features(row):
    img_path = f"../data/imagenet_subset/imagenet-c/{row['corruption']}/{row['severity']}/{row['image_id']}"

    img = tf.keras.utils.load_img(img_path, target_size=(224, 224))
    img_array = tf.keras.utils.img_to_array(img)
    img_array = np.expand_dims(img_array, axis=0) 
    
    features = feat_extractor(img_array, training=False)  # call directly instead of .predict() to stop tf.function retracing

    return features.numpy().flatten().tolist()

tqdm.pandas(desc="Extracting Embeddings of subset")
df_subset['embedding'] = df_subset.progress_apply(extract_features, axis=1) #extract latent embeddings

df_subset.to_parquet("results_card/predictions_with_embeddings.parquet")
print("Saved to results_card/predictions_with_embeddings.parquet")

Extracting Embeddings of subset: 100%|██████████| 1500/1500 [02:10<00:00, 11.47it/s]


Saved to results_card/predictions_with_embeddings.parquet
